# Healthcare MLOps Demo - Part 1: Data Ingestion & Pipeline Setup

**Duration:** ~30 minutes

## Overview
In this notebook, we'll set up the data foundation for our 30-day hospital readmission prediction model:

1. **Internal Stage Data Load** - Load synthetic healthcare data from internal stage
2. **Streams** - Set up CDC (Change Data Capture) for incremental processing
3. **Dynamic Tables** - Build bronze → silver transformation pipeline
4. **Tasks** - Create scheduled orchestration

## Business Context
Hospital readmissions within 30 days are a key quality metric. CMS penalizes hospitals with excess readmissions.
We'll build an ML pipeline to predict readmission risk and enable early intervention.

---
## Setup: Connect to Snowflake

In [ ]:
# Import required packages
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import functions as F
from snowflake.snowpark.types import *
import pandas as pd

# Get active Snowflake session (automatically available in Snowflake Notebooks)
session = get_active_session()

In [ ]:
%%sql -r dataframe_1
USE ROLE ACCOUNTADMIN;
USE DATABASE HEALTHCARE_MLOPS;
USE WAREHOUSE HEALTHCARE_ML_WH;

select concat(
    'Current Role: ', current_role(), '\n',
    'Current Database: ', current_database(), '\n',
    'Current Schema: ', current_schema(), '\n',
    'Current Warehouse: ', current_warehouse(), '\n'
) as context;

---
## 1. Data Ingestion from Internal Stage

We're loading synthetic healthcare data (Synthea-style) that includes:
- **Patients**: Demographics, insurance, location
- **Encounters**: Hospital visits, costs, reasons
- **Conditions**: Diagnoses with ICD-10 codes

> **Note**: Data is stored in an internal stage. The same approach works with external S3/Azure/GCS stages.

In [ ]:
%%sql -r dataframe_2
LIST @HEALTHCARE_MLOPS.STAGING.HEALTHCARE_DATA_STAGE


In [ ]:
%%sql -r dataframe_3
COPY INTO HEALTHCARE_MLOPS.RAW.PATIENTS (
    PATIENT_ID, BIRTH_DATE, DEATH_DATE, SSN, DRIVERS_LICENSE, PASSPORT,
    PREFIX, FIRST_NAME, LAST_NAME, SUFFIX, MAIDEN_NAME, MARITAL_STATUS,
    RACE, ETHNICITY, GENDER, BIRTHPLACE, ADDRESS, CITY, STATE, COUNTY,
    ZIP, LAT, LON, HEALTHCARE_EXPENSES, HEALTHCARE_COVERAGE, INCOME
)
FROM @HEALTHCARE_MLOPS.STAGING.HEALTHCARE_DATA_STAGE/patients/
FILE_FORMAT = (FORMAT_NAME = 'HEALTHCARE_MLOPS.STAGING.CSV_FORMAT')
ON_ERROR = 'CONTINUE';

SELECT COUNT(1) AS patients FROM HEALTHCARE_MLOPS.RAW.PATIENTS;

In [ ]:
%%sql -r dataframe_4
COPY INTO HEALTHCARE_MLOPS.RAW.ENCOUNTERS (
        ENCOUNTER_ID, START_DATETIME, STOP_DATETIME, PATIENT_ID,
        ORGANIZATION_ID, PROVIDER_ID, PAYER_ID, ENCOUNTER_CLASS,
        CODE, DESCRIPTION, BASE_ENCOUNTER_COST, TOTAL_CLAIM_COST,
        PAYER_COVERAGE, REASON_CODE, REASON_DESCRIPTION
    )
    FROM @HEALTHCARE_MLOPS.STAGING.HEALTHCARE_DATA_STAGE/encounters/
    FILE_FORMAT = (FORMAT_NAME = 'HEALTHCARE_MLOPS.STAGING.CSV_FORMAT')
    ON_ERROR = 'CONTINUE';

select count(1) as encounters from HEALTHCARE_MLOPS.RAW.ENCOUNTERS;

In [ ]:
%%sql -r dataframe_5
COPY INTO HEALTHCARE_MLOPS.RAW.CONDITIONS (
    START_DATE, STOP_DATE, PATIENT_ID, ENCOUNTER_ID, CODE, DESCRIPTION
)
FROM @HEALTHCARE_MLOPS.STAGING.HEALTHCARE_DATA_STAGE/conditions/
FILE_FORMAT = (FORMAT_NAME = 'HEALTHCARE_MLOPS.STAGING.CSV_FORMAT')
ON_ERROR = 'CONTINUE';

select count(1) as conditions from HEALTHCARE_MLOPS.RAW.CONDITIONS;

### Preview the Raw Data

In [ ]:
# Preview patients
session.table("HEALTHCARE_MLOPS.RAW.PATIENTS").limit(5).to_pandas()

In [ ]:
# Preview encounters with types
encounters_df = session.table("HEALTHCARE_MLOPS.RAW.ENCOUNTERS")
encounters_df.group_by("ENCOUNTER_CLASS").agg(
    F.count("*").alias("COUNT"),
    F.avg("TOTAL_CLAIM_COST").alias("AVG_COST")
).order_by(F.col("COUNT").desc()).show()

---
## 2. Create Streams for CDC (Change Data Capture)

Streams capture changes to tables, enabling incremental processing of new data.
This is essential for production ML pipelines where we need to process only new records.

In [ ]:
%%sql -r dataframe_6
SHOW STREAMS IN SCHEMA HEALTHCARE_MLOPS.RAW

In [ ]:
%%sql -r dataframe_7
# Check stream status - shows pending changes
SELECT 'Patients Stream has data' as stream_has_data, SYSTEM$STREAM_HAS_DATA('HEALTHCARE_MLOPS.RAW.PATIENTS_STREAM') as value
UNION ALL 
SELECT 'Encounters Stream has data' as stream_has_data,SYSTEM$STREAM_HAS_DATA('HEALTHCARE_MLOPS.RAW.ENCOUNTERS_STREAM') as value;

---
## 3. Dynamic Tables: Bronze → Silver Pipeline

Dynamic Tables automatically maintain materialized views that refresh incrementally.
They're perfect for building data transformation pipelines.

### Pipeline Architecture:
```
RAW (Bronze) → CURATED (Silver) → FEATURE_STORE (Gold)
```

In [ ]:
%%sql -r dataframe_8
-- Dynamic Table 1: Curated Patients (Silver Layer)
  -- Cleanse and standardize patient demographics
  -- Calculate age and risk factors

CREATE OR REPLACE DYNAMIC TABLE HEALTHCARE_MLOPS.CURATED.PATIENTS_CURATED
    TARGET_LAG = '1 hour'
    ROW_TIMESTAMP = TRUE 
    WAREHOUSE = HEALTHCARE_ML_WH
    AS
    SELECT
        PATIENT_ID,
        FIRST_NAME,
        LAST_NAME,
        BIRTH_DATE,
        DEATH_DATE,
        DATEDIFF('year', BIRTH_DATE, COALESCE(DEATH_DATE, LOADED_AT::DATE)) AS AGE,
        GENDER,
        RACE,
        ETHNICITY,
        MARITAL_STATUS,
        CITY,
        STATE,
        ZIP,
        LAT,
        LON,
        HEALTHCARE_EXPENSES,
        HEALTHCARE_COVERAGE,
        INCOME,
        -- Derived: Coverage ratio (financial risk indicator)
        CASE 
            WHEN HEALTHCARE_EXPENSES > 0 
            THEN HEALTHCARE_COVERAGE / HEALTHCARE_EXPENSES 
            ELSE 1.0 
        END AS COVERAGE_RATIO,
        -- Flag deceased patients
        CASE WHEN DEATH_DATE IS NOT NULL THEN TRUE ELSE FALSE END AS IS_DECEASED
    FROM HEALTHCARE_MLOPS.RAW.PATIENTS

In [ ]:
%%sql -r dataframe_9
-- Dynamic Table 2: Curated Encounters (Silver Layer)
  -- Calculate length of stay
  -- Categorize encounter types
  -- Identify high-cost encounters

CREATE OR REPLACE DYNAMIC TABLE HEALTHCARE_MLOPS.CURATED.ENCOUNTERS_CURATED
    TARGET_LAG = '1 hour'
    ROW_TIMESTAMP = TRUE
    WAREHOUSE = HEALTHCARE_ML_WH
    AS
    SELECT
        ENCOUNTER_ID,
        PATIENT_ID,
        START_DATETIME,
        STOP_DATETIME,
        -- Length of stay in hours
        DATEDIFF('hour', START_DATETIME, STOP_DATETIME) AS LENGTH_OF_STAY_HOURS,
        -- Length of stay in days (for inpatient)
        DATEDIFF('day', START_DATETIME, STOP_DATETIME) AS LENGTH_OF_STAY_DAYS,
        ENCOUNTER_CLASS,
        -- Binary flag for inpatient
        CASE WHEN ENCOUNTER_CLASS = 'inpatient' THEN 1 ELSE 0 END AS IS_INPATIENT,
        CODE,
        DESCRIPTION,
        ORGANIZATION_ID,
        PROVIDER_ID,
        PAYER_ID,
        BASE_ENCOUNTER_COST,
        TOTAL_CLAIM_COST,
        PAYER_COVERAGE,
        -- Out of pocket cost
        (TOTAL_CLAIM_COST - PAYER_COVERAGE) AS OUT_OF_POCKET_COST,
        REASON_CODE,
        REASON_DESCRIPTION,
        -- High cost flag (top quartile)
        CASE WHEN TOTAL_CLAIM_COST > 15000 THEN 1 ELSE 0 END AS IS_HIGH_COST,
        LOADED_AT
    FROM HEALTHCARE_MLOPS.RAW.ENCOUNTERS;

In [ ]:
%%sql -r dataframe_10
-- Dynamic Table 3: Curated Conditions (Silver Layer)
  -- Extract ICD-10 category
  -- Flag chronic vs acute conditions

CREATE OR REPLACE DYNAMIC TABLE HEALTHCARE_MLOPS.CURATED.CONDITIONS_CURATED
    TARGET_LAG = '1 hour'
    ROW_TIMESTAMP = TRUE
    WAREHOUSE = HEALTHCARE_ML_WH
    AS
    SELECT
        CONDITION_ID,
        PATIENT_ID,
        ENCOUNTER_ID,
        START_DATE,
        STOP_DATE,
        CODE AS ICD10_CODE,
        -- Extract ICD-10 category (first letter)
        SUBSTR(CODE, 1, 1) AS ICD10_CATEGORY,
        -- Extract ICD-10 subcategory (first 3 chars)
        SUBSTR(CODE, 1, 3) AS ICD10_SUBCATEGORY,
        DESCRIPTION,
        -- Duration in days (NULL if ongoing, use LOADED_AT for ongoing conditions)
        DATEDIFF('day', START_DATE, COALESCE(STOP_DATE, LOADED_AT::DATE)) AS CONDITION_DURATION_DAYS,
        -- Flag chronic conditions (no end date)
        CASE WHEN STOP_DATE IS NULL THEN 1 ELSE 0 END AS IS_CHRONIC,
        -- Major condition categories based on ICD-10
        CASE 
            WHEN CODE LIKE 'I%' THEN 'Circulatory'
            WHEN CODE LIKE 'E%' THEN 'Endocrine'
            WHEN CODE LIKE 'J%' THEN 'Respiratory'
            WHEN CODE LIKE 'C%' THEN 'Neoplasm'
            WHEN CODE LIKE 'M%' THEN 'Musculoskeletal'
            WHEN CODE LIKE 'N%' THEN 'Genitourinary'
            WHEN CODE LIKE 'F%' THEN 'Mental'
            WHEN CODE LIKE 'K%' THEN 'Digestive'
            WHEN CODE LIKE 'G%' THEN 'Nervous'
            WHEN CODE LIKE 'S%' OR CODE LIKE 'T%' THEN 'Injury'
            ELSE 'Other'
        END AS CONDITION_CATEGORY,
        LOADED_AT
    FROM HEALTHCARE_MLOPS.RAW.CONDITIONS;

In [ ]:
%%sql -r dataframe_11
-- SELECT 
--     *
-- FROM TABLE(INFORMATION_SCHEMA.DYNAMIC_TABLES())
-- WHERE SCHEMA_NAME = 'CURATED'
SHOW DYNAMIC TABLES IN SCHEMA CURATED;


---
## 4. Generate Readmission Labels

For ML training, we need labels indicating whether each inpatient encounter 
resulted in a readmission within 30 days.

In [ ]:
%%sql -r dataframe_12
CREATE OR REPLACE DYNAMIC TABLE HEALTHCARE_MLOPS.CURATED.READMISSION_LABELS
    TARGET_LAG = '1 hour'
    WAREHOUSE = HEALTHCARE_ML_WH
    AS
    WITH inpatient_encounters AS (
        SELECT 
            ENCOUNTER_ID,
            PATIENT_ID,
            START_DATETIME,
            STOP_DATETIME,
            -- Get next encounter for same patient
            LEAD(START_DATETIME) OVER (
                PARTITION BY PATIENT_ID 
                ORDER BY START_DATETIME
            ) AS NEXT_ENCOUNTER_START,
            LEAD(ENCOUNTER_CLASS) OVER (
                PARTITION BY PATIENT_ID 
                ORDER BY START_DATETIME
            ) AS NEXT_ENCOUNTER_CLASS
        FROM HEALTHCARE_MLOPS.CURATED.ENCOUNTERS_CURATED
        WHERE ENCOUNTER_CLASS = 'inpatient'
    )
    SELECT
        ENCOUNTER_ID,
        PATIENT_ID,
        STOP_DATETIME AS DISCHARGED_AT,
        -- Calculate days to next encounter
        DATEDIFF('day', STOP_DATETIME, NEXT_ENCOUNTER_START) AS DAYS_TO_NEXT_ENCOUNTER,
        -- Readmission if next inpatient visit within 30 days
        CASE 
            WHEN NEXT_ENCOUNTER_CLASS = 'inpatient' 
                AND DATEDIFF('day', STOP_DATETIME, NEXT_ENCOUNTER_START) <= 30 
                AND DATEDIFF('day', STOP_DATETIME, NEXT_ENCOUNTER_START) >= 0
            THEN TRUE 
            ELSE FALSE 
        END AS READMITTED_WITHIN_30_DAYS,
        CURRENT_TIMESTAMP() AS LABEL_GENERATED_AT
    FROM inpatient_encounters;

In [ ]:
%%sql -r dataframe_13
SELECT 
    COUNT(*) AS TOTAL_INPATIENT_ENCOUNTERS,
    SUM(CASE WHEN READMITTED_WITHIN_30_DAYS THEN 1 ELSE 0 END) AS READMISSIONS,
    ROUND(100.0 * SUM(CASE WHEN READMITTED_WITHIN_30_DAYS THEN 1 ELSE 0 END) / COUNT(*), 2) AS READMISSION_RATE_PCT
FROM HEALTHCARE_MLOPS.CURATED.READMISSION_LABELS

---
## Summary: Data Pipeline Architecture

We've built a complete data ingestion and transformation pipeline:

```
┌─────────────────┐     ┌─────────────────┐     ┌─────────────────┐
│  Internal Stage │────▶│   RAW Tables    │────▶│  Dynamic Tables │
│  (CSV Files)    │     │   (Bronze)      │     │   (Silver)      │
└─────────────────┘     └─────────────────┘     └─────────────────┘
                               │
                               ▼
                        ┌─────────────────┐
                        │    Streams      │
                        │    (CDC)        │
                        └─────────────────┘
```

### Key Objects Created:
- **3 RAW tables**: PATIENTS, ENCOUNTERS, CONDITIONS
- **3 Streams**: For CDC on each table
- **4 Dynamic Tables**: PATIENTS_CURATED, ENCOUNTERS_CURATED, CONDITIONS_CURATED, READMISSION_LABELS
- **1 Task**: Daily data quality check

### Next Steps:
Continue to **Notebook 2: Feature Engineering** to build the Feature Store.

In [ ]:
# Final verification - show all objects
print("=== RAW TABLES ===")
session.sql("SHOW TABLES IN SCHEMA HEALTHCARE_MLOPS.RAW").select('"name"', '"rows"').show()

print("\n=== DYNAMIC TABLES ===")
session.sql("SHOW DYNAMIC TABLES IN SCHEMA HEALTHCARE_MLOPS.CURATED").select('"name"', '"scheduling_state"').show()

print("\n=== STREAMS ===")
session.sql("SHOW STREAMS IN SCHEMA HEALTHCARE_MLOPS.RAW").select('"name"', '"stale"').show()